# 2 · GaSi einmalig einrichten

Dieses Notebook bündelt Paketinstallation, lokale Vorlagen, Zielbereiche, Geräteinventar und
synthetischen Funktionstest. Bereits vorhandene passende Einträge werden wiederverwendet.
**Es lädt keine echten Messdaten hoch und legt keinen echten Prüfling an.**

Beim ersten Start den Projekt-Kernel `.venv` auswählen. Die Paketinstallation ist lokal und
unabhängig vom openBIS-Schreibmodus; bei frisch installierten Paketen danach den Kernel neu starten.
Session auf `False`: geplante Ziele und Aktionen anzeigen. Session auf `True`: Einrichtung ausführen.
Die Zielpfade werden einmalig in `config/playground_test_target.json` gepflegt.


In [ ]:
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Umgebung:", Path(sys.prefix))
if sys.version_info < (3, 10):
    raise RuntimeError("GaSi benötigt Python 3.10 oder neuer. In VS Code einen passenden Kernel auswählen.")
print("Python ist geeignet. Eine Conda-Umgebung oder ein bestimmter Umgebungsname ist nicht erforderlich.")

In [ ]:
from pathlib import Path
import subprocess
import sys

if sys.version_info < (3, 10):
    raise RuntimeError("Für die Projektpakete zuerst einen Kernel mit Python 3.10 oder neuer auswählen.")

# Funktioniert auch, wenn das Notebook aus einem Unterordner geöffnet wurde.
working_dir = Path.cwd().resolve()
repo_root = next((p for p in (working_dir, *working_dir.parents)
                  if (p / "requirements-dev.txt").is_file()
                  and (p / "requirements-playground.txt").is_file()
                  and (p / "requirements-docs.txt").is_file()), None)
if repo_root is None:
    raise RuntimeError("Repository nicht gefunden. Das Notebook im Ordner gasi-openbis öffnen.")

requirements_file = repo_root / "requirements-dev.txt"
print("Installiere Projektpakete für:", sys.executable)
print("Requirements:", requirements_file)
# Entspricht %pip install -r requirements-dev.txt; verwendet sicher den aktiven Kernel.
installation = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)
print("Installation abgeschlossen. Kernel neu starten und anschließend die Prüfzelle unten ausführen.")

## Ziele einmalig prüfen

Vorhandene lokale Konfigurationen bleiben erhalten. Bei einer neuen Arbeitskopie die beiden
Collection-Pfade im Editor eintragen und lokal speichern. Der Editor ändert keine Serverdaten.
Der tatsächliche Standort der Geräte steht unter `gasi_inventory.location` in derselben JSON.
Die Einrichtung ergänzt aus diesen Projekten die Mess-, Auswertungs- und Prüflingsziele.
Es gibt keine weiteren Freigabeschalter oder Bestätigungsphrasen.


In [ ]:
import gasi_workflow
from gasi_playground_targets import show_target_editor

print("Neue lokale Vorlagen:", gasi_workflow.initialize_local())
show_target_editor()


In [ ]:
import gasi_workflow

result, message = gasi_workflow.setup_project()
gasi_workflow.show_result(result, message)


## Datenmodell und HTML neu erzeugen
Diese beiden Zellen bleiben auch nach der Einrichtung nutzbar: offline, ohne openBIS.
Nach einer Modelländerung nur diese Erzeugungszelle erneut ausführen.


In [ ]:
import importlib
import gasi_model_html, gasi_concept_html, gasi_workflow
from IPython.display import FileLink, display

importlib.reload(gasi_model_html)
importlib.reload(gasi_concept_html)
for label, path in gasi_workflow.generate_model().items():
    print(label)
    display(FileLink(str(path)))
